In [2]:
# 如果想要对某个list排序，但是不想改变原list，返回的应该是indices
init_dis = [3,5,7,4,0,-1]
indices = sorted(range(len(init_dis)), key=lambda i: init_dis[i])
for i in indices:
    print(f"{i}:{init_dis[i]}", end=",")
print("")
reverse_indices = sorted(range(len(init_dis)), key=lambda i: -init_dis[i])
for i in reverse_indices:
    print(f"{i}:{init_dis[i]}", end=",")

5:-1,4:0,0:3,3:4,1:5,2:7,
2:7,1:5,3:4,0:3,4:0,5:-1,

In [6]:
init_dis = [3,5,7,4,0,-1]
print(sum(init_dis))
# print(sum(1,2,3,4,5,6)) # 报错，因为sum如果不给可迭代对象，就需要给两个参数
from functools import reduce
def add(x, y) :            # 两数相加
    return x + y
sum1 = reduce(add, [1,2,3,4,5])   # 计算列表和：1+2+3+4+5
sum2 = reduce(lambda x, y: x+y, [1,2,3,4,5])  # 使用 lambda 匿名函数
print(sum1)
print(sum2)

18
15
15


In [16]:
# 不用numpy实现手撕kmeans算法
import random
import math

def calculate_dis(p1, p2):
    # 计算欧氏距离
    return math.sqrt(sum((a-b)**2 for a,b in zip(p1, p2)))

def k_means(data, k, max_iters=100, tol=1e-4):
    # 初始化
    centers = random.sample(data, k)
    for iteration in range(max_iters):
        # k个簇，用于存放属于该簇的点
        clusters = [[] for _ in range(k)]
        # 分配阶段
        for point in data:
            distances = [calculate_dis(point, c) for c in centers]
            # 找最近中心点的索引
            closest_idx = distances.index(min(distances))
            clusters[closest_idx].append(point)
        # 记录旧中心用于计算偏移
        old_centers = list(centers)

        # 更新
        for i in range(k):
            if not clusters[i]:
                # 预防空聚类：如果这个簇没有点，随机重新指派
                centers[i] = random.choice(data)
                continue
            # 计算均值作为新的中心
            num_points = len(clusters[i])
            dims = len(data[0])
            new_center = []
            for d in range(dims):
                dim_sum = sum(p[d] for p in clusters[i])
                new_center.append(dim_sum / num_points)
            centers[i] = new_center

        # 判断收敛
        total_shift = sum(calculate_dis(centers[i], old_centers[i]) for i in range(k))
        if total_shift < tol:
            print(f"在第{iteration}次迭代时提前收敛。")
            break
    return centers, clusters

if __name__ == "__main__":
    test_data = [[1,2], [1,1], [-1,3], [5,8], [-4,-4], [3,-5], [5,-2]]
    K = 3
    final_centers, final_clusters = k_means(test_data, K)
    for i,c in enumerate(final_centers):
        print(f"簇{i}中心：{c}，包含点数：{len(final_clusters[i])}")

在第2次迭代时提前收敛。
簇0中心：[-0.75, 0.5]，包含点数：4
簇1中心：[5.0, 8.0]，包含点数：1
簇2中心：[4.0, -3.5]，包含点数：2


In [18]:
# Kmeans，使用numpy
import numpy as np

def k_means(data, k, max_iters=100, tol=1e-4):
    data = np.array(data)
    n_samples, n_features = data.shape
    indices = np.random.choice(n_samples, k, replace=False)
    centers = data[indices]

    for i in range(max_iters):
        distances = np.linalg.norm(data[:, np.newaxis] - centers, axis=2)
        labels = np.argmin(distances, axis=1)
        new_centers = np.array([
            data[labels==j].mean(axis=0) if len(data[labels==j]) > 0
            else data[np.random.choice(n_samples)]
            for j in range(k)
        ])
        center_shift = np.linalg.norm(new_centers - centers)
        if center_shift < tol:
            print(f"在第{i}次迭代时提前收敛。")
            break
        centers = new_centers
    return centers, labels

if __name__ == "__main__":
    # 生成 1000 个 2 维随机点
    X = np.concatenate([
        np.random.randn(500, 2) + [2, 2],
        np.random.randn(500, 2) + [-2, -2]
    ])
    
    centers, labels = k_means(X, k=5)
    unique_labels, counts = np.unique(labels, return_counts=True)
    count_dict = dict(zip(unique_labels, counts))
    for i,c in enumerate(centers):        
        # print(f"簇{i}中心：{c}，包含点数：{np.sum(labels==i)}")
        print(f"簇{i}中心：{c}，包含点数：{count_dict.get(i, 0)}")

在第12次迭代时提前收敛。
簇0中心：[-1.11698669 -2.52124204]，包含点数：160
簇1中心：[-2.47096153 -1.27225406]，包含点数：152
簇2中心：[-2.88045322 -2.6472556 ]，包含点数：124
簇3中心：[2.01566323 2.08806213]，包含点数：491
簇4中心：[-1.10624206 -0.34563636]，包含点数：73
